In [1]:
#!git clone https://github.com/whyhardt/SPICE.git

In [2]:
# !pip install -e SPICE

In [6]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional

from spice import SpiceEstimator, SpiceConfig, csv_to_dataset, BaseModel, plot_session, split_data_along_sessiondim
from spice.precoded import workingmemory

sys.path.append("../../..") 
from weinhardt2026.utils.benchmarking_gru import Model as GRU, training as training_gru
from weinhardt2026.studies.castro2025.benchmarking_castro2025 import Castro2025Model, training as training_castro

# For custom RNN
import torch

# print current path
print("Current path:", os.getcwd())

Current path: /home/daniel/repositories/SPICE/weinhardt2026/studies/castro2025


will be pipeline notebook for castro2025 benchmark model

## Load dataset

Let's load the data first with the `csv_to_dataset` method. This method returns a `SpiceDataset` object which we can use right away 

In [2]:
# Load your data, should be in SPICE/weinhardt2025/data/eckstein2024/eckstein2024.csv
path_file = 'data/eckstein2024.csv'
# headers of csv: participant,block,trial_id,choice,reward,rt,payout_1,payout_2,payout_3,payout_4

dataset = csv_to_dataset(
    file = path_file,
    df_participant_id='participant',
    df_choice='choice',
    df_feedback='reward',
    df_block='block',
    )

# Optional reduced dataset for quick sanity checks
use_reduced_data = False
max_sessions = 8
max_trials = 100

if use_reduced_data:
    xs_small = dataset.xs[:max_sessions, :max_trials]
    ys_small = dataset.ys[:max_sessions, :max_trials]
    dataset = dataset.__class__(xs_small, ys_small, device=dataset.device)

# structure of dataset:
# dataset has two main attributes: xs -> inputs; ys -> targets (next action)
# shape: (n_participants*n_blocks*n_experiments, n_timesteps, features)
# features are (n_actions * action, n_actions * reward, n_additional_inputs * additional_input, time_trial, trials, block_number, experiment_id, participant_id)

# in order to set up the participant embedding we have to compute the number of unique participants in our data 
# to get the number of participants n_participants we do:
n_participants = len(dataset.xs[..., -1].unique())
n_actions = dataset.ys.shape[-1]

print(f"Shape of dataset: {dataset.xs.shape}")
print(f"Number of participants: {n_participants}")
print(f"Number of actions in dataset: {n_actions}")
print(f"Number of additional inputs: {dataset.xs.shape[-1]-2*n_actions-5}")
print(f"Number of sessions: {len(dataset.xs[..., -3].unique())}")

Shape of dataset: torch.Size([4158, 150, 1, 15])
Number of participants: 862
Number of actions in dataset: 5
Number of additional inputs: 0
Number of sessions: 5


In [3]:
test_sessions = (2, 3)  # pick sessions that exist for all participants; adjust if needed
dataset_train, dataset_test = split_data_along_sessiondim(dataset, test_sessions)

## SPICE Setup

## SPICE Training

Let's setup now the `SpiceEstimator` object and fit it to the data! 

We are going to do this in two steps:

1. Without fitting the SINDy coefficients to get the pure RNN performance given the selected architecture. 
2. With fitting SINDy coefficients to get the final performance of the interpretable model

That way we can disentangle the gap between GRU and SPICE w.r.t. architecture and SINDy library 

In [ ]:
# fitting without SINDy coefficients

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=workingmemory.SpiceModel,
        spice_config=workingmemory.CONFIG,
        n_actions=n_actions,
        n_participants=n_participants,

        epochs= 1000,
        
        sindy_weight=0,
        sindy_library_polynomial_degree=2,
        sindy_pruning_frequency=100,
        sindy_ensemble_pruning=0.05,
        sindy_threshold_pruning=0.01,
        sindy_alpha=0.0001,
        
        verbose=True,
    )
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

 55%|██████████████████████████████████▌                            | 549/1000 [22:37<18:35,  2.47s/it, L(Train)=0.5434864, L(Val,RNN)=0.5663379, Conv=2.63e-03]


In [ ]:
path_spice = 'params/spice_castro2025.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=workingmemory.SpiceModel,
        spice_config=workingmemory.CONFIG,
        n_actions=n_actions,
        n_participants=n_participants,
        n_experiments=1,
        
        epochs=1000,
        warmup_steps=200,
        learning_rate=0.01,
        batch_size=None,
        ensemble_size=10,
        
        sindy_weight=0.1,
        sindy_alpha=0.0001,
        sindy_pruning_frequency=100,
        sindy_threshold_pruning=0.05,
        sindy_ensemble_pruning=0.01,
        sindy_library_polynomial_degree=2,

        verbose=True,
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        save_path_spice=path_spice,
    )

In [ ]:
estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

100%|██████████████████████████████████████████| 10/10 [01:55<00:00, 11.57s/it, L(Train)=0.4058124, L(Val,RNN)=0.3898913, L(Val,SINDy)=0.4134763, Conv=6.75e-03]
----------------------------------------------------------------------------------------------------------------------------------------------------------------
SPICE Model (Coefficients: 17):
value_reward_chosen[t+1]     = 0.173 1 + 0.849 value_reward_chosen[t] + 0.153 reward[t] + 0.154 reward[t-1] + 0.099 reward[t-2] + 0.029 reward[t-3] 
value_reward_not_chosen[t+1] = -0.09 1 + 0.886 value_reward_not_chosen[t] + 0.209 reward[t-1] + 0.11 reward[t-2] + -0.082 reward[t-3] 
value_choice[t+1]            = 0.005 1 + 0.94 value_choice[t] + 0.488 choice[t] + -0.0 choice[t-1] + -0.211 choice[t-2] + -0.252 choice[t-3] 
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Pruning patience:
value_reward_chosen:     0, 0, 0, 0, 0, 0

100%|██████████| 1000/1000 [00:19<00:00, 51.39it/s, loss=0.0009329, n_params=17.00+/-0.00]



Training results:
	L(Train, RNN): 0.4058124
	L(Val, RNN):   0.3898913
	L(Val, SINDy): 0.3977097

RNN training finished.
Training took 135.86 seconds.
Saving SPICE model to ../params/castro2025/spice_castro2025.pkl...


In [11]:
estimator.load_spice(path_spice)

In [12]:
# Print example SPICE model for first participant
print("\nExample SPICE model (participant 0):")
estimator.print_spice_model(participant_id=0)


Example SPICE model (participant 0):
value_reward_chosen[t+1]     = 0.056 1 + 0.819 value_reward_chosen[t] + 0.304 reward[t] + 0.248 reward[t-1] + 0.112 reward[t-2] + 0.017 reward[t-3] 
value_reward_not_chosen[t+1] = -0.112 1 + 0.853 value_reward_not_chosen[t] + 0.33 reward[t-1] + 0.096 reward[t-2] + -0.04 reward[t-3] 
value_choice[t+1]            = -0.0 1 + 0.707 value_choice[t] + 0.527 choice[t] + 0.077 choice[t-1] + -0.145 choice[t-2] + -0.247 choice[t-3] 


## Benchmarking

### Castro2025 benchmark model

In [ ]:
# Benchmark model: Castro et al. 2025
cfs = Castro2025Model(
    n_participants=n_participants,
    n_actions=n_actions,
    batch_first=True,
    )

path_castro = path_spice.replace('spice_', 'cfs_')

In [ ]:
epochs = 10 #00

def to_time_major(tensor, name):
    if tensor.dim() == 4:
        if tensor.shape[2] != 1:
            raise ValueError(
                f"{name} has within-trial dimension W={tensor.shape[2]} (>1); Castro2025 expects one step per trial."
            )
        tensor = tensor.squeeze(2)
    elif tensor.dim() != 3:
        raise ValueError(f"{name} must be 3D or 4D, got shape {tuple(tensor.shape)}")
    return tensor.permute(1, 0, 2)  # (T, B, F)

xs_train = to_time_major(dataset_train.xs, "dataset_train.xs")
ys_train = to_time_major(dataset_train.ys, "dataset_train.ys")
xs_test = to_time_major(dataset_test.xs, "dataset_test.xs") if dataset_test is not None else None
ys_test = to_time_major(dataset_test.ys, "dataset_test.ys") if dataset_test is not None else None

cfs = training_castro(
    model=cfs,
    xs=xs_train,
    ys=ys_train,
    xs_test=xs_test,
    ys_test=ys_test,
    epochs=epochs,
    lr=0.01,
    max_restarts=1,
    )

torch.save(cfs.state_dict(), path_castro)

Weight decoupling enabled in AdaBelief
Rectification enabled in AdaBelief


  Restart 1: loss=1.24488 (max epochs), best=1.24488


In [5]:
optimizer_cfs = torch.optim.Adam(params=cfs.parameters(), lr=0.01)
epochs = 1000

cfs = training_gru(
    model=cfs,
    optimizer=optimizer_cfs,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
)

Epoch 1/1000: L(Train): 0.7041839957237244; L(Test): 0.7037692666053772
Epoch 2/1000: L(Train): 0.7028644680976868; L(Test): 0.7026912569999695
Epoch 3/1000: L(Train): 0.7015863656997681; L(Test): 0.7016575932502747
Epoch 4/1000: L(Train): 0.7003492116928101; L(Test): 0.7006667256355286
Epoch 5/1000: L(Train): 0.6991519331932068; L(Test): 0.6997163891792297
Epoch 6/1000: L(Train): 0.6979917883872986; L(Test): 0.6988044381141663
Epoch 7/1000: L(Train): 0.6968685388565063; L(Test): 0.6979275345802307
Epoch 8/1000: L(Train): 0.6957809329032898; L(Test): 0.6970831751823425
Epoch 9/1000: L(Train): 0.6947261095046997; L(Test): 0.6962699890136719
Epoch 10/1000: L(Train): 0.6937036514282227; L(Test): 0.6954859495162964
Epoch 11/1000: L(Train): 0.6927114129066467; L(Test): 0.694729745388031
Epoch 12/1000: L(Train): 0.691747784614563; L(Test): 0.6939992308616638
Epoch 13/1000: L(Train): 0.6908119916915894; L(Test): 0.6932937502861023
Epoch 14/1000: L(Train): 0.6899011135101318; L(Test): 0.692611

In [ ]:
cfs.load_state_dict(torch.load(path_castro, map_location='cpu'))

<All keys matched successfully>

### GRU Model

In [8]:
gru = GRU(n_actions)

path_gru = path_spice.replace('spice_', 'gru_')

In [9]:
epochs = 1000
optimizer_gru = torch.optim.Adam(gru.parameters(), lr=0.01)

gru = training_gru(
    model=gru,
    optimizer=optimizer_gru,
    dataset_train=dataset_train,
    dataset_test=dataset_test,
    epochs=epochs,
    ).to(torch.device('cpu'))

torch.save(gru.state_dict(), path_gru)

Epoch 1/1000: L(Train): 1.6765508651733398; L(Test): 1.6016274690628052
Epoch 2/1000: L(Train): 1.6044373512268066; L(Test): 1.5383278131484985
Epoch 3/1000: L(Train): 1.5410804748535156; L(Test): 1.4792242050170898
Epoch 4/1000: L(Train): 1.4819362163543701; L(Test): 1.4219510555267334
Epoch 5/1000: L(Train): 1.4250223636627197; L(Test): 1.3652276992797852
Epoch 6/1000: L(Train): 1.3687645196914673; L(Test): 1.3086081743240356
Epoch 7/1000: L(Train): 1.3126821517944336; L(Test): 1.2518980503082275
Epoch 8/1000: L(Train): 1.2570552825927734; L(Test): 1.1946552991867065
Epoch 9/1000: L(Train): 1.2004213333129883; L(Test): 1.1366262435913086
Epoch 10/1000: L(Train): 1.1434540748596191; L(Test): 1.0782496929168701
Epoch 11/1000: L(Train): 1.0862507820129395; L(Test): 1.0207358598709106
Epoch 12/1000: L(Train): 1.0294196605682373; L(Test): 0.9656042456626892
Epoch 13/1000: L(Train): 0.9752727746963501; L(Test): 0.9146974682807922
Epoch 14/1000: L(Train): 0.9252979755401611; L(Test): 0.8703

In [19]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))

<All keys matched successfully>

# ANALYSIS

In [10]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation
from weinhardt2026.analysis.analysis_coefficients_distributions import analysis_coefficients_distributions
from weinhardt2026.analysis.analysis_coefficients_individuals import analysis_coefficients_individuals

## General Analysis

In [21]:
# Dataset-specific behavioral analysis placeholder.
# Replace with Eckstein2024-specific columns if needed.

In [ ]:
print("Fitted Castro2025 parameters:")
print("\nBeta_r")
print(cfs.beta_r)
print("\nLapse")
print(cfs.lapse)
print("\nPrior")
print(cfs.prior)
print("\nAlpha exploration rate")
print(cfs.alpha_exploration_rate)
print("\nDecay rate")
print(cfs.decay_rate)
print("\nGamma")
print(cfs.gamma)
print("\nTemperature")
print(cfs.temperature)

Fitted Castro2025 parameters:

Beta_r
tensor([0.6951, 0.6931], grad_fn=<ClampBackward1>)

Lapse
tensor([0.4986, 0.5000], grad_fn=<ClampBackward1>)

Prior
tensor([0.6914, 0.6931], grad_fn=<ClampBackward1>)

Alpha exploration rate
tensor([0.5000, 0.5000], grad_fn=<ClampBackward1>)

Decay rate
tensor([0.5008, 0.5000], grad_fn=<ClampBackward1>)

Gamma
tensor([0.6916, 0.6931], grad_fn=<SoftplusBackward0>)

Temperature
tensor([0.6912, 0.6931], grad_fn=<ClampBackward1>)


## Analysis Model Evaluation

In [11]:
analysis_model_evaluation(
    dataset=dataset_test,
    # spice_model=estimator,
    benchmark_model=cfs.to(torch.device('cpu')),
    gru_model=gru.to(torch.device('cpu')),
    )

Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...


,Trial Lik.,(std),n_parameters,(std),NLL,(std),AIC,BIC
Benchmark,0.561617,0.160108,13.0,0.0,0.576936,0.293696,1.328369,1.590457
GRU,0.575591,0.158565,1893.0,0.0,0.552358,0.299084,26.514147,64.678368
SPICE-RNN,1.000000,0.000000,NaN,0.0,0.000000,0.000000,0.000000,0.000000
SPICE,1.000000,0.000000,NaN,0.0,0.000000,0.000000,0.000000,0.000000


## Analysis coefficient distributions

In [ ]:
analysis_coefficients_distributions(
    spice_model=estimator,
    output_dir='results',
)

Extracting coefficient data...
  Ensemble=1, Participants=2, Experiments=1, Modules=3, Total terms=17

ANALYSIS 1: Ensemble Consistency
  Mean presence agreement: 1.000
  Mean presence rate:      1.000
  Ensemble spread plots saved.
  Ensemble CV heatmaps saved.

ANALYSIS 2: Coefficient Distributions
  Terms with >50% presence: 17 / 17
  Terms with 0% presence:   0 / 17
  Violin plots saved.
  Presence rate bar chart saved.
  Experiment comparison skipped (X=1).
  Sparsity heatmaps saved.

All results saved to: ../data/castro2025/


(                     module                     term  term_index  \
 0       value_reward_chosen                        1           0   
 1       value_reward_chosen      value_reward_chosen           1   
 2       value_reward_chosen                reward[t]           2   
 3       value_reward_chosen              reward[t-1]           3   
 4       value_reward_chosen              reward[t-2]           4   
 5       value_reward_chosen              reward[t-3]           5   
 6   value_reward_not_chosen                        1           0   
 7   value_reward_not_chosen  value_reward_not_chosen           1   
 8   value_reward_not_chosen              reward[t-1]           2   
 9   value_reward_not_chosen              reward[t-2]           3   
 10  value_reward_not_chosen              reward[t-3]           4   
 11             value_choice                        1           0   
 12             value_choice             value_choice           1   
 13             value_choice      

## Analysis Individual Differences

In [ ]:
# analysis_coefficients_individuals(
#     criterion="SomeCriterionColumnInYourDataset",
#     analysis="disc",  # also: "cont"
#     reference="ReferenceGroupFromCriterionColumn",  # only necessary if analysis="disc"
    
#     path_data=path_file,
    
#     spice_model=estimator,
    
#     dir_output='results',
# )

'analysis_coefficients_individuals(\n    criterion="SomeCriterionColumnInYourDataset",\n    analysis="disc",  # also: "cont"\n    reference="ReferenceGroupFromCriterionColumn",  # only necessary if analysis="disc"\n\n    path_data=file,\n\n    spice_model=estimator,\n)'